# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [1]:
ROLL_NO = 1024170007          # <-- your full roll number
NAME = "Alla Harshith"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [3]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [4]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170007   Name: Alla Harshith   Fingerprint: C2355A09   Tickets: 14

Ticket_ID                                                                                   Ticket_Text
      T01      I keep getting logged out of the student portal after the latest update. This is urgent.
      T02                                                Re-installing didn't help; log-in still fails!
      T03                                           The WiFi was working yesterday. Can you check this?
      T04                                                                 Why isn't the portal opening?
      T05                                                  Call me on +91 98765 43210 or 0175-239-3201.
      T06                          The student portal was working yesterday. Is there another solution?
      T07 Password keeps asking for login when connected to campus Wi-Fi. It works on another computer.
      T08           I keep getting logged out of the password while uploading the f

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [5]:
# Your code
import pandas as pd

# Your generated dataset
tickets = {
    "T01": "I keep getting logged out of the student portal after the latest update. This is urgent.",
    "T02": "Re-installing didn't help; log-in still fails!",
    "T03": "The WiFi was working yesterday. Can you check this?",
    "T04": "Why isn't the portal opening?",
    "T05": "Call me on +91 98765 43210 or 0175-239-3201.",
    "T06": "The student portal was working yesterday. Is there another solution?",
    "T07": "Password keeps asking for login when connected to campus Wi-Fi. It works on another computer.",
    "T08": "I keep getting logged out of the password while uploading the file. This is urgent.",
    "T09": "Python 3.12 was installed yesterday.",
    "T10": "The lms was working yesterday. Please resolve it.",
    "T11": "The student portal has stopped syncing. I have tried several times!",
    "T12": "The server stopped responding. I don't know what changed.",
    "T13": "The server failed to open. This is urgent.",
    "T14": "I am unable to use the Wi-Fi when connected to campus Wi-Fi. This is urgent."
}

processed = {}
rows = []

for ticket_id, text in tickets.items():
    result = process_ticket(text)
    processed[ticket_id] = result

    rows.append({
        "Ticket_ID": ticket_id,
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique_Tokens": len(set(result["tokens"])),
        "Final_Tokens": len(result["final"])
    })

df_q1 = pd.DataFrame(rows)

# Total row for count-based columns
total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": df_q1["Sentences"].sum(),
    "Tokens": df_q1["Tokens"].sum(),
    "Punctuation": df_q1["Punctuation"].sum(),
    "Stopwords": df_q1["Stopwords"].sum(),
    "Unique_Tokens": df_q1["Unique_Tokens"].sum(),
    "Final_Tokens": df_q1["Final_Tokens"].sum()
}

df_q1 = pd.concat(
    [df_q1, pd.DataFrame([total_row])],
    ignore_index=True
)

display(df_q1)

# Detailed display for one selected ticket
selected = "T01"

print("\nSelected Ticket:", selected)
print("Text:", tickets[selected])
print("\nActual tokens:")
print(processed[selected]["tokens"])

print("\nPunctuation tokens:")
print(processed[selected]["punct"])

print("\nStopwords:")
print(processed[selected]["stops"])

print("\nFinal tokens:")
print(processed[selected]["final"])

,Ticket_ID,Sentences,Tokens,Punctuation,Stopwords,Unique_Tokens,Final_Tokens
0,T01,2,18,2,8,16,8
1,T02,1,9,2,1,9,6
2,T03,2,11,2,5,11,4
3,T04,1,7,1,3,7,3
4,T05,1,9,1,3,9,5
5,T06,2,12,2,4,12,6
6,T07,2,17,2,5,16,10
7,T08,2,17,2,8,15,7
8,T09,1,6,1,1,6,4
9,T10,2,10,2,3,9,5



Selected Ticket: T01
Text: I keep getting logged out of the student portal after the latest update. This is urgent.

Actual tokens:
['i', 'keep', 'getting', 'logged', 'out', 'of', 'the', 'student', 'portal', 'after', 'the', 'latest', 'update', '.', 'this', 'is', 'urgent', '.']

Punctuation tokens:
['.', '.']

Stopwords:
['i', 'out', 'of', 'the', 'after', 'the', 'this', 'is']

Final tokens:
['keep', 'getting', 'logged', 'student', 'portal', 'latest', 'update', 'urgent']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [6]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Your code
import re

# -------------------------------
# REGEX PATTERNS
# -------------------------------

# Words longer than 5 letters
long_word_pattern = r"\b[A-Za-z]{6,}\b"

# Numbers including decimals and versions
# Examples: 95.5, 2.0.1, 3.12
number_pattern = r"\b\d+(?:\.\d+)+\b|\b\d+\b"

# Capitalized words
capitalized_pattern = r"\b[A-Z][A-Za-z]*\b"

# Words beginning with a vowel
vowel_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"

# Email
email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

# URL
url_pattern = r"(?:https?://|www\.)[^\s;,]+"

# Phone numbers
phone_pattern = r"(?:\+91[\s-]?\d{5}[\s-]?\d{5}|\d{4}-\d{3}-\d{4})"


# -------------------------------
# EXTRACTION ON RAW TICKETS
# -------------------------------

for ticket_id, text in tickets.items():

    print("\n", "=" * 60)
    print(ticket_id, ":", text)

    print("Words > 5 letters:",
          re.findall(long_word_pattern, text))

    print("Numbers:",
          re.findall(number_pattern, text))

    print("Capitalized words:",
          re.findall(capitalized_pattern, text))

    print("Words beginning with vowel:",
          re.findall(vowel_pattern, text))


# -------------------------------
# REPLACEMENT FUNCTION
# -------------------------------

def replace_special(text):

    text = re.sub(email_pattern, "<EMAIL>", text)

    text = re.sub(url_pattern, "<URL>", text)

    text = re.sub(phone_pattern, "<PHONE>", text)

    return text


print("\n\n===== REPLACEMENT ON TICKETS =====")

for ticket_id, text in tickets.items():
    print(ticket_id, ":", replace_special(text))


# -------------------------------
# TEST ON PROVIDED TEST_LINES
# -------------------------------

print("\n\n===== TEST_LINES =====")

for line in TEST_LINES:

    print("\nOriginal:")
    print(line)

    print("Words > 5 letters:",
          re.findall(long_word_pattern, line))

    print("Numbers:",
          re.findall(number_pattern, line))

    print("Capitalized words:",
          re.findall(capitalized_pattern, line))

    print("Vowel words:",
          re.findall(vowel_pattern, line))

    print("After replacement:")
    print(replace_special(line))


T01 : I keep getting logged out of the student portal after the latest update. This is urgent.
Words > 5 letters: ['getting', 'logged', 'student', 'portal', 'latest', 'update', 'urgent']
Numbers: []
Capitalized words: ['I', 'This']
Words beginning with vowel: ['I', 'out', 'of', 'after', 'update', 'is', 'urgent']

T02 : Re-installing didn't help; log-in still fails!
Words > 5 letters: ['installing']
Numbers: []
Capitalized words: ['Re']
Words beginning with vowel: ['installing', 'in']

T03 : The WiFi was working yesterday. Can you check this?
Words > 5 letters: ['working', 'yesterday']
Numbers: []
Capitalized words: ['The', 'WiFi', 'Can']
Words beginning with vowel: []

T04 : Why isn't the portal opening?
Words > 5 letters: ['portal', 'opening']
Numbers: []
Capitalized words: ['Why']
Words beginning with vowel: ['isn', 'opening']

T05 : Call me on +91 98765 43210 or 0175-239-3201.
Words > 5 letters: []
Numbers: ['91', '98765', '43210', '0175', '239', '3201']
Capitalized words: ['Call']

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [7]:
# Your code
from nltk.tokenize import RegexpTokenizer, word_tokenize

TOKEN_PATTERN = r"""(?x)
    https?://[^\s;,]+
    | www\.[^\s;,]+
    | [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}
    | \+91[\s-]?\d{5}[\s-]?\d{5}
    | \d{4}-\d{3}-\d{4}
    | \d+(?:\.\d+)+
    | [A-Za-z]+(?:[-'][A-Za-z]+)+
    | [A-Za-z]+
    | \d+
    | [^\w\s]
"""

my_regexp_tokenizer = RegexpTokenizer(TOKEN_PATTERN)


def my_tokenize(text):
    return my_regexp_tokenizer.tokenize(text)


# Test on TEST_LINES
print("===== TEST_LINES =====")

for i, line in enumerate(TEST_LINES, 1):

    print(f"\nTEST LINE {i}")

    print("\nmy_tokenize():")
    print(my_tokenize(line))

    print("\nword_tokenize():")
    print(word_tokenize(line))


# Run on our tickets
print("\n\n===== TICKETS =====")

for ticket_id, text in tickets.items():
    print("\n", ticket_id)
    print(my_tokenize(text))

===== TEST_LINES =====

TEST LINE 1

my_tokenize():
['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']

word_tokenize():
['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']

TEST LINE 2

my_tokenize():
['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1', ',', '250', '.']

word_tokenize():
['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']

TEST LINE 3

my_tokenize():
['The', 'state-of-the-art', 'lab', "isn't", 'open', ';', 'log-in', 'at', '5.30', 'p', '.', 'm', '.', "didn't", 'work', '.']

word_tokenize():
['The', 'state-of-the-art', 'lab', 'is', "n't", 'open', ';', 'log-in', 'at

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [8]:
# Your code
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import RegexpStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import pandas as pd

porter = PorterStemmer()
lancaster = LancasterStemmer()

# Our dataset contains words such as:
# uploading -> upload
# installing -> install
# working -> work
#
# Therefore we use common suffixes ing/ed.
regexp_stemmer = RegexpStemmer(
    r"(ing|ed)$",
    min=4
)

lemmatizer = WordNetLemmatizer()


# Collect unique final tokens
unique_final = sorted(set(
    token
    for result in processed.values()
    for token in result["final"]
))


def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith("J"):
        return wordnet.ADJ

    elif treebank_tag.startswith("V"):
        return wordnet.VERB

    elif treebank_tag.startswith("N"):
        return wordnet.NOUN

    elif treebank_tag.startswith("R"):
        return wordnet.ADV

    else:
        return wordnet.NOUN


# POS tagging
tagged_words = pos_tag(unique_final)

rows = []

for word, tag in tagged_words:

    porter_result = porter.stem(word)

    lancaster_result = lancaster.stem(word)

    regexp_result = regexp_stemmer.stem(word)

    lemma_result = lemmatizer.lemmatize(
        word,
        get_wordnet_pos(tag)
    )

    outputs = {
        word,
        porter_result,
        lancaster_result,
        regexp_result,
        lemma_result
    }

    # Keep only words where outputs differ
    if len(outputs) > 1:

        rows.append({
            "Word": word,
            "POS": tag,
            "Porter": porter_result,
            "Lancaster": lancaster_result,
            "Regexp": regexp_result,
            "POS-aware Lemma": lemma_result
        })


stem_df = pd.DataFrame(rows)

display(stem_df)

,Word,POS,Porter,Lancaster,Regexp,POS-aware Lemma
0,another,DT,anoth,anoth,another,another
1,asking,VBG,ask,ask,ask,ask
2,call,NN,call,cal,call,call
3,campus,NN,campu,camp,campus,campus
4,changed,VBD,chang,chang,chang,change
5,computer,NN,comput,comput,computer,computer
6,connected,VBD,connect,connect,connect,connect
7,failed,VBD,fail,fail,fail,fail
8,fails,NNS,fail,fail,fails,fails
9,file,VB,file,fil,file,file


**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [9]:
# Your code
from collections import Counter
from nltk.util import bigrams
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------
# BUILD CORPORA
# ----------------------------

S1 = [
    token
    for result in processed.values()
    for token in result["tokens"]
]

S2 = [
    token
    for result in processed.values()
    for token in result["tokens"]
    if not is_punct(token)
]

S3 = [
    token
    for result in processed.values()
    for token in result["final"]
]

S4 = [
    porter.stem(token)
    for token in S3
]


# ----------------------------
# CORPUS STATISTICS
# ----------------------------

def corpus_stats(tokens):

    total = len(tokens)
    unique = len(set(tokens))

    ttr = unique / total if total else 0

    return total, unique, ttr


stats = []

for name, corpus in [
    ("S1 - All tokens", S1),
    ("S2 - Without punctuation", S2),
    ("S3 - Final tokens", S3),
    ("S4 - Porter stemmed", S4)
]:

    total, unique, ttr = corpus_stats(corpus)

    stats.append({
        "Corpus": name,
        "Total Tokens": total,
        "Unique Tokens": unique,
        "TTR": round(ttr, 4)
    })


stats_df = pd.DataFrame(stats)

display(stats_df)

,Corpus,Total Tokens,Unique Tokens,TTR
0,S1 - All tokens,168,86,0.5119
1,S2 - Without punctuation,143,82,0.5734
2,S3 - Final tokens,82,56,0.6829
3,S4 - Porter stemmed,82,52,0.6341


**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:
*